# Baseline: XGBoost + RepeatedStratifiedKFold

## Методология победителя

- **Модель:** XGBoost
- **CV:** RepeatedStratifiedKFold (n_repeats=3, n_splits=5)
- **Стратификация:** `(target * 400).astype(int16)`
- **OOF predictions** для инкремента в Advanced

In [ ]:
# Импорт
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_predict
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import r2_score
from scipy.stats.mstats import winsorize
from tqdm.auto import tqdm
import os
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

print('✅ Библиотеки загружены')

In [ ]:
# Загрузка данных
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

X_full = train.drop(columns=['id', 'FloodProbability'])
y_full = train['FloodProbability']
X_test = test.drop(columns=['id'])

print(f'Train: {train.shape}, Test: {test.shape}')

In [ ]:
# Winsorization
def winsorize_features(df, limits=(0.01, 0.01)):
    df_win = df.copy()
    for col in tqdm(df.columns, desc='Winsorization'):
        df_win[col] = winsorize(df[col], limits=limits)
    return df_win

X_full_win = winsorize_features(X_full)
X_test_win = winsorize_features(X_test)

In [ ]:
# RobustScaler
scaler = RobustScaler()
X_full_scaled = pd.DataFrame(scaler.fit_transform(X_full_win), columns=X_full_win.columns, index=X_full_win.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_win), columns=X_test_win.columns, index=X_test_win.index)

print('✅ Scaling done')

In [ ]:
# Feature Engineering (baseline)
def generate_baseline_features(df_orig, df_scaled):
    features = df_scaled.copy()
    features['sum'] = df_orig.sum(axis=1)
    features['mean'] = df_orig.mean(axis=1)
    features['std'] = df_orig.std(axis=1)
    features['max'] = df_orig.max(axis=1)
    features['min'] = df_orig.min(axis=1)
    features['median'] = df_orig.median(axis=1)
    features['range'] = features['max'] - features['min']
    features['q25'] = df_orig.quantile(0.25, axis=1)
    features['q75'] = df_orig.quantile(0.75, axis=1)
    features['iqr'] = features['q75'] - features['q25']
    features['cv'] = features['std'] / (features['mean'] + 1e-10)
    return features

X_full_feat = generate_baseline_features(X_full_win, X_full_scaled)
X_test_feat = generate_baseline_features(X_test_win, X_test_scaled)

print(f'✅ Features: {X_full_feat.shape[1]} (исходные {X_full.shape[1]} → +{X_full_feat.shape[1] - X_full.shape[1]})')

In [ ]:
# XGBoost модель
xgb_model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42,
    n_jobs=-1,
    tree_method='hist'
)

print('✅ Модель создана: XGBoostRegressor')

In [ ]:
# Создаем папку для сохранения
os.makedirs('baseline_output', exist_ok=True)

# RepeatedStratifiedKFold OOF predictions
print('\n' + '='*70)
print('ОБУЧЕНИЕ С RepeatedStratifiedKFold (3x5)')
print('='*70)

n_repeats, n_splits = 3, 5
rskf = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=42)

# Стратификация по дискретизованному таргету
y_discrete = (y_full * 400).astype(np.int16)

# OOF predictions
oof_preds = np.zeros((len(X_full_feat), n_repeats))
test_preds = np.zeros((len(X_test_feat), n_repeats * n_splits))

fold_idx = 0
for repeat in range(n_repeats):
    print(f'\nRepeat {repeat+1}/{n_repeats}')
    
    for fold, (train_idx, val_idx) in enumerate(rskf.split(X_full_feat, y_discrete)):
        if fold >= n_splits * (repeat + 1):
            break
        if fold < n_splits * repeat:
            continue
            
        X_train, X_val = X_full_feat.iloc[train_idx], X_full_feat.iloc[val_idx]
        y_train, y_val = y_full.iloc[train_idx], y_full.iloc[val_idx]
        
        # Обучение
        xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        
        # OOF
        oof_preds[val_idx, repeat] += xgb_model.predict(X_val)
        
        # Test predictions
        test_preds[:, fold_idx] = xgb_model.predict(X_test_feat)
        
        # R² score
        val_r2 = r2_score(y_val, xgb_model.predict(X_val))
        print(f'  Fold {(fold % n_splits)+1}: R² = {val_r2:.6f}')
        
        fold_idx += 1
    
    # Repeat OOF R²
    repeat_r2 = r2_score(y_full, oof_preds[:, repeat])
    print(f'  → Repeat {repeat+1} OOF R² = {repeat_r2:.6f}')

# Финальный OOF (среднее по repeats)
oof_final = oof_preds.mean(axis=1)
final_r2 = r2_score(y_full, oof_final)

print(f'\n{'='*70}')
print(f'🏆 FINAL OOF R² = {final_r2:.6f}')
print('='*70)

In [ ]:
# Сохранение для Advanced (инкремент)
np.save('baseline_output/baseline_predictions_train_oof.npy', oof_final)
np.save('baseline_output/baseline_predictions_test.npy', test_preds.mean(axis=1))

print('\n✅ OOF predictions сохранены для инкремента в Advanced')

In [ ]:
# Submission
submission = pd.DataFrame({
    'id': test['id'],
    'FloodProbability': test_preds.mean(axis=1)
})

submission.to_csv('submission_baseline_xgb.csv', index=False)
print(f'\n✅ Submission: submission_baseline_xgb.csv')
print(f'   R² (OOF): {final_r2:.6f}')